# Capítulo 4: A Matemática dos Sistemas Biológicos
## Exemplos Práticos de Simulação e Modelagem em Python

Este notebook reúne os exemplos práticos discutidos no Capítulo 4 da disciplina **Bioinformática para Biologia de Sistemas**. Aqui você encontrará implementações de:
1. Modelos discretos (Mapa Logístico com ruído)
2. Integração numérica de EDOs acopladas (scipy.integrate)
3. Modelo Lotka-Volterra (Predador-Presa) e Retratos de Fase
4. Análise de estabilidade linear (Jacobiano simbólico e autovalores)
5. Diagramas de Bifurcação (Pitchfork)
6. Ciclos limite (Oscilador de Van der Pol)
7. Análise de sensibilidade local por perturbação
8. Análise de sensibilidade global (Método de Sobol usando a biblioteca SALib)

--- 
## Exemplo 1: Simulação de Sistemas Discretos (Mapa Logístico com Ruído)

O mapa logístico é um modelo clássico de crescimento populacional discreto. Aqui simulamos a versão determinística:
$$x_{t+1} = r x_t (1 - x_t)$$

E a versão estocástica com ruído multiplicativo:
$$x_{t+1} = r x_t (1 - x_t) e^{\eta_t}$$
onde $\eta_t \sim \mathcal{N}(0, \sigma^2)$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Parâmetros do Mapa Logístico
r = 3.8       # Comportamento caótico
sigma = 0.02  # Intensidade do ruído estocástico
steps = 50
x_det = np.zeros(steps)
x_est = np.zeros(steps)
x_det[0] = x_est[0] = 0.5

# Simulação Recursiva
for t in range(steps - 1):
    # Determinista
    x_det[t+1] = r * x_det[t] * (1 - x_det[t])
    # Estocástico com ruído multiplicativo (truncado em 0 e 1)
    noise = np.random.normal(0, sigma)
    val = r * x_est[t] * (1 - x_est[t]) * np.exp(noise)
    x_est[t+1] = np.clip(val, 0.0, 1.0)

# Visualizar
plt.figure(figsize=(10, 4))
plt.plot(x_det, 'b-o', label='Determinista')
plt.plot(x_est, 'r--x', label='Estocástico')
plt.xlabel('Tempo (t)')
plt.ylabel('População x(t)')
plt.legend()
plt.grid(True)
plt.show()

--- 
## Exemplo 2: Resolvendo EDOs Acopladas com SciPy

Abaixo integramos numericamente um sistema genérico de duas EDOs acopladas usando `scipy.integrate.odeint`.

In [ ]:
import numpy as np
from scipy.integrate import odeint
import matplotlib.pyplot as plt

# Definir sistema de EDOs
def sistema(X, t, alpha, beta):
    """Sistema de duas variáveis acopladas"""
    x, y = X
    dxdt = alpha*x - beta*x*y
    dydt = -y + x*y
    return [dxdt, dydt]

# Parâmetros
alpha = 1.0
beta = 0.5

# Condição inicial
X0 = [2.0, 1.0]

# Tempo
t = np.linspace(0, 20, 1000)

# Resolver
sol = odeint(sistema, X0, t, args=(alpha, beta))

# Plotar série temporal
plt.figure(figsize=(10, 4))
plt.plot(t, sol[:, 0], 'b-', label='x(t)')
plt.plot(t, sol[:, 1], 'r-', label='y(t)')
plt.xlabel('Tempo')
plt.ylabel('Concentração')
plt.legend()
plt.grid(True)
plt.show()

--- 
## Exemplo 3: Simulando o Modelo de Lotka-Volterra e Retratos de Fase

O modelo clássico de Lotka-Volterra descreve a dinâmica de populações predador-presa. Abaixo mostramos a dinâmica cíclica em termos de trajetórias no retrato de fase.

In [ ]:
import numpy as np
from scipy.integrate import odeint
import matplotlib.pyplot as plt

def lotka_volterra(X, t, alpha, beta, delta, gamma):
    """Modelo predador-presa"""
    x, y = X
    dxdt = alpha*x - beta*x*y
    dydt = delta*x*y - gamma*y
    return [dxdt, dydt]

# Parâmetros
alpha, beta, delta, gamma = 1.0, 0.5, 0.5, 1.0

# Condições iniciais
X0 = [2.0, 1.0]

# Tempo
t = np.linspace(0, 50, 1000)

# Resolver
sol = odeint(lotka_volterra, X0, t, args=(alpha, beta, delta, gamma))

# Plotar retrato de fase
plt.figure(figsize=(8, 6))
plt.plot(sol[:, 0], sol[:, 1], 'b-', linewidth=2, label='Trajetória')
plt.plot(gamma/delta, alpha/beta, 'ro', markersize=10, label='Equilíbrio (Foco)')
plt.xlabel('Presas (x)')
plt.ylabel('Predadores (y)')
plt.legend()
plt.grid(True)
plt.show()

--- 
## Exemplo 4: Calculando Jacobiano e Autovalores Simbolicamente com SymPy

Para analisar a estabilidade de pontos fixos em sistemas não-lineares, calculamos a matriz Jacobiana e seus autovalores no ponto de equilíbrio.

In [ ]:
import numpy as np
from scipy.linalg import eig
import sympy as sp

# Definir variáveis simbólicas
x, y = sp.symbols('x y')

# Definir sistema
f1 = x*(1 - x - 0.5*y)
f2 = y*(-0.75 + 0.5*x)

# Calcular Jacobiano simbolicamente
J = sp.Matrix([[sp.diff(f1, x), sp.diff(f1, y)],
               [sp.diff(f2, x), sp.diff(f2, y)]])

print("Matriz Jacobiana Simbólica:")
sp.pprint(J)

# Encontrar pontos de equilíbrio
equilibria = sp.solve([f1, f2], [x, y])
print("\nPontos de equilíbrio encontrados:", equilibria)

# Avaliar numericamente no ponto de equilíbrio não-trivial (x*=1.5, y*=1)
x_star, y_star = 1.5, 1.0
J_numeric = np.array(J.subs([(x, x_star), (y, y_star)])).astype(float)

print(f"\nJacobiano numérico no equilíbrio ({x_star}, {y_star}):\n", J_numeric)

# Calcular autovalores
eigenvalues, eigenvectors = eig(J_numeric)
print(f"\nAutovalores em ({x_star}, {y_star}):", eigenvalues)
print("Estável?", all(np.real(eigenvalues) < 0))

--- 
## Exemplo 5: Gerando Diagramas de Bifurcação (Pitchfork Supercrítica)

Uma bifurcação descreve a mudança qualitativa na estabilidade e número de pontos fixos conforme um parâmetro varia. Abaixo varremos $\mu$ para construir o diagrama da bifurcação pitchfork supercrítica:
$$\frac{dx}{dt} = \mu x - x^3$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import fsolve

def pitchfork(x, mu):
    """Bifurcação pitchfork supercrítica"""
    return mu*x - x**3

# Varrer parâmetro mu
mu_vals = np.linspace(-2, 2, 100)
equilibria = []

for mu in mu_vals:
    # Encontrar todos os equilíbrios aproximados a partir de diferentes chutes iniciais
    eq_list = []
    for x0 in [-2, 0, 2]:
        eq = fsolve(pitchfork, x0, args=(mu,))[0]
        if abs(pitchfork(eq, mu)) < 1e-6:
            eq_list.append(eq)

    # Remover equilíbrios duplicados
    eq_list = list(set(np.round(eq_list, 6)))

    for eq in eq_list:
        # Verificar estabilidade linear: df/dx = mu - 3*x^2
        dfdx = mu - 3*eq**2
        if dfdx < 0:
            equilibria.append((mu, eq, 'stable'))
        else:
            equilibria.append((mu, eq, 'unstable'))

# Separar os ramos estáveis e instáveis
stable = [(m, x) for m, x, s in equilibria if s == 'stable']
unstable = [(m, x) for m, x, s in equilibria if s == 'unstable']

# Plotar
plt.figure(figsize=(8, 5))
plt.plot([m for m, x in stable], [x for m, x in stable], 'b.', markersize=6, label='Estável')
plt.plot([m for m, x in unstable], [x for m, x in unstable], 'r.', markersize=4, label='Instável')
plt.axvline(0, color='k', linestyle=':', alpha=0.5, label='Ponto Crítico (\mu_c = 0)')
plt.xlabel('Parâmetro $\mu$')
plt.ylabel('Equilíbrio $x^*$')
plt.title('Diagrama de Bifurcação Pitchfork Supercrítica')
plt.legend()
plt.grid(True)
plt.show()

--- 
## Exemplo 6: Simulação de Ciclos Limite (Oscilador de Van der Pol)

Diferente de sistemas lineares (onde oscilações periódicas requerem condições iniciais perfeitas e são frágeis), sistemas não-lineares podem possuir ciclos limite atratores e estruturalmente robustos.

In [ ]:
import numpy as np
from scipy.integrate import odeint
import matplotlib.pyplot as plt

def van_der_pol(X, t, mu):
    """Oscilador de Van der Pol"""
    x, y = X
    dxdt = y
    dydt = mu*(1 - x**2)*y - x
    return [dxdt, dydt]

# Parâmetros (intensidade da não-linearidade)
mu = 2.0

# Condição inicial (longe do ciclo limite)
X0 = [0.1, 0.1]

# Tempo
t = np.linspace(0, 50, 2000)

# Resolver
sol = odeint(van_der_pol, X0, t, args=(mu,))

# Plotar série temporal e retrato de fase
plt.figure(figsize=(12, 5))

# Série temporal
plt.subplot(1, 2, 1)
plt.plot(t, sol[:, 0], 'b-', label='x(t)')
plt.xlabel('Tempo')
plt.ylabel('Estado (x)')
plt.title('Série Temporal')
plt.grid(True)
plt.legend()

# Retrato de fase mostrando convergência para o Ciclo Limite
plt.subplot(1, 2, 2)
plt.plot(sol[:, 0], sol[:, 1], 'r-', linewidth=1.2, label='Trajetória')
plt.plot(sol[0, 0], sol[0, 1], 'go', markersize=8, label='Cond. Inicial')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Retrato de Fase (Ciclo Limite)')
plt.legend()
plt.grid(True)
plt.show()

--- 
## Exemplo 7: Análise de Sensibilidade Local por Perturbação

A sensibilidade local mede como a saída do modelo em estado estacionário varia sob pequenas perturbações em parâmetros individuais:
$$S_i = \frac{\partial y}{\partial p_i} \frac{p_i}{y} \approx \frac{\Delta y / y}{\Delta p_i / p_i}$$

In [ ]:
import numpy as np
from scipy.integrate import odeint

def modelo_sistema(x, t, Vmax, Km, kdeg, S):
    """Modelo simples de síntese induzida por substrato S e degradação de primeira ordem"""
    dxdt = (Vmax * S) / (Km + S) - kdeg * x
    return dxdt

# Parâmetros nominais
params_nom = {'Vmax': 10, 'Km': 5, 'kdeg': 0.5, 'S': 8}

# Calcular sensibilidade local por diferença finita centralizada
def calc_sensibilidade(param_name, delta=0.01):
    """Calcula coeficiente de sensibilidade local normalizado"""
    t = np.linspace(0, 50, 500)
    x0 = 0.0
    
    # Simulação base (nominal)
    params_base = params_nom.copy()
    sol_base = odeint(modelo_sistema, x0, t,
                      args=(params_base['Vmax'], params_base['Km'],
                            params_base['kdeg'], params_base['S']))
    y_base = sol_base[-1, 0]  # Estado estacionário
    
    # Simulação perturbada (+)
    params_pert = params_base.copy()
    params_pert[param_name] *= (1 + delta)
    sol_pert = odeint(modelo_sistema, x0, t,
                      args=(params_pert['Vmax'], params_pert['Km'],
                            params_pert['kdeg'], params_pert['S']))
    y_pert = sol_pert[-1, 0]
    
    # Sensibilidade normalizada
    S_val = ((y_pert - y_base) / y_base) / delta
    return S_val

# Calcular para cada parâmetro relevante
print("Coeficientes de Sensibilidade Local Normalizados:")
for param in ['Vmax', 'Km', 'kdeg']:
    S = calc_sensibilidade(param)
    print(ff"S_{{x, {param}}} = {S:.3f}")

--- 
## Exemplo 8: Análise de Sensibilidade Global (Método de Sobol)

A análise global explora todo o espaço de parâmetros simultaneamente, estimando os índices de Sobol de primeira ordem ($S_1$ - efeito direto) e total ($S_T$ - efeito total incluindo interações).

In [ ]:
try:
    from SALib.sample import saltelli
    from SALib.analyze import sobol
except ImportError:
    print("SALib não está instalada! Instalando agora usando pip...")
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "SALib"])
    from SALib.sample import saltelli
    from SALib.analyze import sobol

import numpy as np

# Definir problema de sensibilidade global
problem = {
    'num_vars': 3,
    'names': ['Vmax', 'Km', 'kdeg'],
    'bounds': [[5, 15],    # Vmax nominal = 10, variação: [5, 15]
               [2.5, 7.5], # Km nominal = 5, variação: [2.5, 7.5]
               [0.25, 0.75]] # kdeg nominal = 0.5, variação: [0.25, 0.75]
}

# Gerar amostras (método Saltelli para cálculo de Sobol)
n_samples = 512
param_values = saltelli.sample(problem, n_samples)

# Avaliar o modelo para cada conjunto de parâmetros
# Y conterá o estado estacionário do modelo: x_ss = (Vmax * S) / (kdeg * (Km + S)) com S = 8
Y = np.zeros([param_values.shape[0]])
S_val = 8.0

for i, params in enumerate(param_values):
    Vmax, Km, kdeg = params
    # Solução analítica do estado estacionário
    x_ss = (Vmax * S_val) / (kdeg * (Km + S_val))
    Y[i] = x_ss

# Executar análise de Sobol
Si = sobol.analyze(problem, Y)

print("Índices de Sobol de Primeira Ordem (Efeitos Principais S1):")
for name, s1 in zip(problem['names'], Si['S1']):
    print(f"  S1_{name} = {s1:.4f}")

print("\nÍndices de Sobol de Efeito Total (ST):")
for name, st in zip(problem['names'], Si['ST']):
    print(f"  ST_{name} = {st:.4f}")